# 51 — 既存M4 runの安全な再開

`50_m4_derivatives.ipynb` から作成済みの M4 run を、人間が fresh kernel から再確認・再開するための入口です。`50` は run identity にハッシュ固定されているため編集しません。

このNotebookは、新しい永続保存先を黙って承認しません。既存の `.validated_rg_persistent_root.json` が `/storage/validated_4d_su2_rg` と一致する場合だけ、kernel内の環境変数を復元します。markerがない場合は計算開始前に停止します。


In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'AGENTS.md').is_file():
    raise RuntimeError(f'Project rootを特定できません: {PROJECT_ROOT}')
sys.path.insert(0, str(PROJECT_ROOT))

from src.common import read_json
from src.runtime import (
    PERSIST_ACK_ENV, PERSIST_ACK_TOKEN, PERSIST_ROOT_ENV,
    environment_info, validate_persistent_root,
)

PERSIST_ROOT_CHOICE = Path('/storage/validated_4d_su2_rg').resolve()
marker_path = PERSIST_ROOT_CHOICE / '.validated_rg_persistent_root.json'
if not marker_path.is_file():
    raise RuntimeError(
        f'承認済み永続markerがありません: {marker_path}。'
        '新しい保存先を自動承認せず、READMEの永続保存確認に従ってください。'
    )
marker = read_json(marker_path)
if marker.get('schema_version') != 1 or marker.get('resolved_root') != str(PERSIST_ROOT_CHOICE):
    raise RuntimeError(f'永続markerと保存先が一致しません: {marker}')

os.environ[PERSIST_ROOT_ENV] = str(PERSIST_ROOT_CHOICE)
os.environ[PERSIST_ACK_ENV] = PERSIST_ACK_TOKEN
os.environ['VALIDATED_RG_M3_RUN_ID'] = 'M3-20260720T013551Z-ae995e91e861'
os.environ['VALIDATED_RG_M4_RUN_ID'] = 'M4-20260720T021737Z-b9c9514fed11'
PERSIST_ROOT = validate_persistent_root()
print(json.dumps({
    'persistent_root': str(PERSIST_ROOT),
    'persistent_marker': marker,
    'M3_run_id': os.environ['VALIDATED_RG_M3_RUN_ID'],
    'M4_run_id': os.environ['VALIDATED_RG_M4_RUN_ID'],
    '判定': '承認済み永続保存先を検証しました。M4再開可。',
}, ensure_ascii=False, indent=2))


In [ ]:
from src.m4_config import M4Config
from src.m4_orchestrator import create_or_resume_m4
from src.m4_parent import verify_accepted_m3_parent

M4_CONFIG = M4Config()
M3_EVIDENCE = verify_accepted_m3_parent(PROJECT_ROOT, M4_CONFIG)
m4_orchestrator = create_or_resume_m4(
    PERSIST_ROOT, M4_CONFIG, PROJECT_ROOT,
    run_id=os.environ['VALIDATED_RG_M4_RUN_ID'],
    test_report=None,
)
if m4_orchestrator.session_policy != 'RESUMED_STANDARD_FIVE_HOUR_THIRTY_LIMIT':
    raise RuntimeError(f'再開時間ポリシーが不正です: {m4_orchestrator.session_policy}')
M4_RESULT = m4_orchestrator.run_until_checkpoint()
if M4_RESULT['certification_status'] != 'NOT_CERTIFIED':
    raise RuntimeError('M4 certification invariantが破られました。')
print(json.dumps(M4_RESULT, ensure_ascii=False, indent=2))


In [ ]:
loaded = m4_orchestrator.checkpoints.load_latest(restore_rng=False)
if loaded is None:
    raise RuntimeError('有効なM4 checkpointがありません。')
report_path = m4_orchestrator.run_root / 'reports/M4_report.json'
report = read_json(report_path) if report_path.is_file() else None
inspection = {
    'run_id': loaded.state.run_id,
    'phase': loaded.state.phase,
    'checkpoint': str(loaded.path),
    'checkpoint_index': loaded.state.checkpoint_index,
    'done': sum(item.status == 'done' for item in loaded.queue.items.values()),
    'pending': sum(item.status == 'pending' for item in loaded.queue.items.values()),
    'skipped_invalid': list(loaded.skipped_invalid),
    'session_policy': m4_orchestrator.session_policy,
    'milestone_status': None if report is None else report['milestone_status'],
    'missing_bound_terms': None if report is None else report['missing_bound_terms'],
    'certification_status': loaded.state.certification_status,
}
print(json.dumps(inspection, ensure_ascii=False, indent=2))
if report is not None:
    assert report['phase'] == 'M4_COMPLETE'
    assert report['milestone_status'] == 'BLOCKED_MATH'
    assert report['certification_status'] == 'NOT_CERTIFIED'
    print('判定: M4実装完了、BLOCKED_MATH。M5へ進行不可。')
else:
    print('M4未完了です。同じrun IDで次回もこのNotebookを再実行してください。')


## 時間制限

このNotebookからの既存run再開は通常ポリシーです。15分周期とwork-item境界で保存し、5時間以降は長い仕事を始めず、5時間15分でdrain、5時間20分でfinal save、5時間30分で必ず戻ります。現在のrunは `M4_COMPLETE` なので、通常はハッシュ検証だけを行い再計算しません。
